# CIFAR-10 dataset class

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from torchvision.transforms import ToTensor

class MyCIFAR10Dataset(Dataset):
    def __init__(self, train=True):
        split = "train" if train else "test"
        self.dataset = load_dataset(
            "uoft-cs/cifar10",
            split=split,
        )
        self.to_tensor = ToTensor()

        self.index_to_class = dict(enumerate(self.dataset.features["label"].names))

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        example = self.dataset[index]

        image = self.to_tensor(example["img"])  # [3, 32, 32]
        label = example["label"]

        return image, label


# Our training, validation, and test datasets

In [ ]:
from torch.utils.data import Subset
import random

full_train_dataset = MyCIFAR10Dataset(train=True)
test_dataset = MyCIFAR10Dataset(train=False)

indices = list(range(len(full_train_dataset)))
random.Random(42).shuffle(indices)

# We will use 10000 examples for training
train_dataset = Subset(full_train_dataset, indices[:10000])

# And 1000 examples for validation
validation_dataset = Subset(full_train_dataset, indices[-1000:])

# Dataloaders

In [ ]:

mini_batch_size = 32

# Define our dataloaders for training, validation, and testing:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=mini_batch_size,
    shuffle=True,
)
validation_dataloader = DataLoader(
    validation_dataset,
    batch_size=mini_batch_size,
    shuffle=False,
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=mini_batch_size,
    shuffle=False,
)

In [ ]:
len(train_dataset), len(validation_dataloader), len(test_dataset)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for index, ax in enumerate(axes.flat):
    image, label = test_dataset[index]

    ax.imshow(image.permute(1, 2, 0), cmap="gray")
    ax.set_title(f"{test_dataset.index_to_class[label]}")
    ax.axis("off")

plt.tight_layout()
plt.show()

# Feedforward neural network

In [ ]:
from torch import nn


class FeedforwardNeuralNetwork(nn.Module):
    def __init__(self, in_features, hidden_features, out_features, num_hidden_layers):
        super().__init__()

        layers = []

        for layer_index in range(num_hidden_layers):
            
            input_size = in_features if layer_index == 0 else hidden_features

            layers.append(nn.Linear(input_size, hidden_features))
            layers.append(nn.ReLU())  # This is the activation function applied after each hidden layer

        layers.append(nn.Linear(hidden_features, out_features))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Training

In [ ]:
import torch.nn.functional as F
from tqdm.auto import tqdm

# Define our hyperparameters:
number_of_epochs = 5
learning_rate = 0.001

# Lets increase our mini-batch size to 32 for the training set:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
)

# Define our model:
model = FeedforwardNeuralNetwork(3072, 32, 10, 2)

# Define our optimiser:
optimiser = torch.optim.SGD(model.parameters(), lr=learning_rate)

# Train the model:
for epoch in range(number_of_epochs):

    model.train()  # Set the model to training mode (enables training features)
    for mini_batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{number_of_epochs}"):

        input, target = mini_batch

        output = model(torch.flatten(input, start_dim=1))  # Flatten 3x32x32 images to 3072 element vectors for feedforward neural network input. Is this needed for the CNN?

        loss = F.cross_entropy(output, target)
        
        loss.backward()
        optimiser.step()
        optimiser.zero_grad()

    # Evaluate the model on the validation dataset:
    model.eval()  # Set the model to evaluation mode (disables training features)

    val_accuracy = 0
    val_loss = 0
    for mini_batch in validation_dataloader:
        input, target = mini_batch


        with torch.no_grad():  # Disable gradient computation for evaluation

            output = model(torch.flatten(input, start_dim=1))  # Flatten 3x32x32 images to 3072-dimensional vectors for feedforward neural network input. Is this needed for the CNN?

            predictions = torch.argmax(output, dim=1)
            val_accuracy += (predictions == target).sum().item()
            val_loss += F.cross_entropy(output, target).item()

    val_accuracy /= len(validation_dataset)
    val_loss /= len(validation_dataloader)
    print(f"Validation accuracy for epoch {epoch + 1}: {val_accuracy:.4f}")
    print(f"Validation loss for epoch {epoch + 1}: {val_loss:.4f}")    

# Testing

In [ ]:
# Testing:
model.eval()  # Set the model to evaluation mode (disables training features)

test_accuracy = 0
test_loss = 0
for batch in test_dataloader:
    input, target = batch

    with torch.no_grad():  # Disable gradient computation for evaluation

        output = model(torch.flatten(input, start_dim=1))  # Flatten 3x32x32 images to 3072-dimensional vectors for feedforward neural network input. Is this needed for the CNN?

        predictions = torch.argmax(output, dim=1)
        test_accuracy += (predictions == target).sum().item()
        test_loss += F.cross_entropy(output, target).item()
        
test_accuracy /= len(test_dataset)
test_loss /= len(test_dataloader)
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test loss: {test_loss:.4f}")

# Visualisation

In [ ]:
model.eval()

test_dataloader = DataLoader(
    test_dataset,
    batch_size=10,
    shuffle=False,
)

x, y = next(iter(test_dataloader))

with torch.no_grad():
    z = model(torch.flatten(x, start_dim=1))  # Flatten 3x32x32 images to 3072-dimensional vectors for feedforward neural network input.
    predictions = z.argmax(dim=1)

number_to_show = 10

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for index, axis in enumerate(axes.flat):
    image = x[index] # Images were flattened from [28, 28] to [784].


    axis.imshow(image.permute(1, 2, 0), cmap="gray")
    axis.set_title(
        f"Predicted: {test_dataset.index_to_class[predictions[index].item()]}\n"
        f"Actual: {test_dataset.index_to_class[y[index].item()]}\n",
        color="green" if predictions[index] == y[index] else "red",
    )
    axis.axis("off")

plt.tight_layout()
plt.show()

# Challenge: Who can get the highest CIFAR-10 test accuracy?

My best test accuracy is: ?